# Tutorial 10-1: The Volatility of k-means
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Introduction: Clustering as Unsupervised Learning
As discussed in lecture, clustering is a form of **Unsupervised Learning**. Unlike classification, we do not have target labels ($y$) to guide the model. Instead, we look for an underlying structure in the data based solely on the features ($x$).

The **k-means algorithm** is one of the most widely used clustering techniques. It attempts to partition $n$ observations into $k$ clusters in which each observation belongs to the cluster with the nearest mean (centroid). 

### The Objective
This tutorial demonstrates two primary challenges in using k-means:
1. **Random Initialization:** How the algorithm's outcome depends on where the centroids start.
2. **Model Selection (Choosing K):** How to use the Elbow Method and Silhouette Analysis to choose the right number of clusters when you cannot see the data in 2D.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

# Set a random seed for consistent data generation
np.random.seed(42)

# Generate synthetic data with 4 distinct clusters (centers)
X, y_true = make_blobs(n_samples=500, centers=4, cluster_std=0.60, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, color='gray', alpha=0.5)
plt.title("Original Unlabeled Data\n(Visually we can see 4 clusters, but the algorithm doesn't know this yet)")
plt.xlabel("Feature $x_1$")
plt.ylabel("Feature $x_2$")
plt.show()

## 2. The Problem of Local Optima
The k-means algorithm follows a **Coordinate Descent** optimization approach:
1. **Assignment Step:** Assign each point to its nearest centroid ($c^{(i)}$).
2. **Update Step:** Recalculate the centroid ($\mu_j$) as the mean of its assigned points.

Because k-means minimizes a non-convex distortion function (Sum of Squared Errors - SSE), it is highly sensitive to the initial positions of the centroids. If two centroids start too close to each other, they might 'split' a single natural cluster, while another centroid is forced to 'cover' two natural clusters.

The probability of a perfect initialization (getting exactly one centroid in each cluster) is very low for large $k$ ($p = k!/k^k$).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# seeds that result in different outcomes
for i, seed in enumerate([2, 5]):
    # n_init=1 forces the algorithm to run only once with one set of random centroids.
    # Usually, sklearn runs 10 times and picks the best one by default.
    km = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed)
    y_km = km.fit_predict(X)
    
    ax[i].scatter(X[:, 0], X[:, 1], c=y_km, s=30, cmap='viridis', alpha=0.7)
    ax[i].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], 
                  s=250, marker='*', c='red', edgecolor='black', label='Centroids')
    
    ax[i].set_title(f"Run {i+1} (Random Seed {seed})\nTotal SSE: {km.inertia_:.2f}")
    ax[i].legend()
    
plt.show()

### Observation: Sub-optimal Clustering
In the plots above, notice how the **SSE (Inertia)** differs. The run with the lower SSE is mathematically better. 

**Solutions demonstrated:**
* **Multiple Restarts:** In practice, we run k-means many times with different initializations and pick the one that yields the lowest SSE.
* **k-means++:** (Optional internal detail) Modern libraries like Scikit-Learn use `init='k-means++'` by default, which selects initial centroids that are spread far apart, greatly reducing the risk of poor convergence.

## 3. Selecting K: The Elbow Method
The most common question in clustering is: *How many clusters should I use?*

If we plot the **Distortion Function (J)**, also known as SSE or Inertia, against the number of clusters ($k$), we will see that SSE always decreases as $k$ increases. This is because having more centroids allows them to be closer to their assigned points. 

However, we reach a point of **Diminishing Returns**. We look for the 'bend' or 'elbow' where adding another cluster doesn't improve the SSE significantly. This bend often represents the most natural grouping of the data.

In [ ]:
sse = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    sse.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, sse, marker='o', linestyle='--', color='blue')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Distortion (SSE / Inertia)')
plt.title('Elbow Plot: SSE vs. Number of Clusters')

# Highlight the elbow
plt.annotate('Natural structure at k=4',
             xy=(4, sse[3]), xytext=(6, sse[1]),
             arrowprops=dict(facecolor='black', shrink=0.05),
             fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Validating with the Silhouette Coefficient
The **Silhouette Coefficient** provides a more nuanced way to evaluate 'goodness' of fit. It combines two ideas discussed in lecture:
1. **Cohesion (a):** How close a point is to others in its own cluster.
2. **Separation (b):** How far a point is from the next closest cluster.

The coefficient is calculated as: $s = (b - a) / max(a, b)$

* **Close to 1:** The point is well-clustered.
* **Close to 0:** The point is on the boundary between two clusters.
* **Negative:** The point may have been assigned to the wrong cluster.

In [ ]:
sil_scores = []
k_range = range(2, 11) # Silhouette requires comparing clusters, so k starts at 2

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    y_km = km.fit_predict(X)
    score = silhouette_score(X, y_km)
    sil_scores.append(score)
    print(f"For k={k}, Mean Silhouette Score: {score:.4f}")

plt.figure(figsize=(10, 6))
plt.bar(k_range, sil_scores, color='purple', alpha=0.6)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis: Higher is Better')
plt.axhline(y=max(sil_scores), color='red', linestyle='--')
plt.show()

## Summary and Practical Tips

### 1. Handling Initialization
Never rely on a single run of k-means. Always use multiple restarts (`n_init`) and leverage the `k-means++` algorithm to choose smarter starting points.

### 2. Interpreting Plots
* **The Elbow Method** is an excellent first pass, but the elbow isn't always sharp. In noisy data, the curve might look very smooth.
* **Silhouette Analysis** provides a quantitative peak. If your silhouette score is very low (e.g., < 0.2), it suggests that your clusters are overlapping or that the data doesn't naturally cluster into spherical shapes.

### 3. The Human Element
As noted in the slides, 'Clusters are in the eye of the beholder.' Statistics help us justify our choices, but clustering remains a 'black art' that requires domain knowledge to interpret correctly.